In [2]:
from pymilvus import MilvusClient
client = MilvusClient(uri="http://localhost:19530")

print("所有数据库：", client.list_databases())     # 确认有 rag_tutorial

client.use_database("rag_tutorial")                # 切到正确的库

print("该库所有集合：", client.list_collections())  # 确认集合名，大概率是 docs


所有数据库： ['default', 'rag_demo', 'rag_tutorial']
该库所有集合： ['docs']


In [11]:
import os
client.use_database("rag_tutorial")              # 确保在正确的库里

data = client.query(
    collection_name="docs",                       # ← 表名，按 list 结果改
    filter="id >= 0",
    output_fields=["text", "source"],             # ← 别用 "*"，见下
    limit=10000
)

output_path = r"E:\Milvus\rag_docs.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print(f"已导出 {len(data)} 条")


已导出 13 条


In [4]:
from pymilvus import MilvusClient
client = MilvusClient(uri="http://localhost:19530")
client.use_database("rag_tutorial")

In [5]:
print("当前状态:", client.get_load_state(collection_name="docs"))

当前状态: {'state': <LoadState: Loading>, 'progress': 50}


In [6]:
client.release_collection(collection_name="docs")
client.load_collection(collection_name="docs")     # ← 重点看这步报不报错、报什么错

KeyboardInterrupt: 

In [7]:
import time
from pymilvus import MilvusClient

client = MilvusClient(uri="http://localhost:19530")
client.use_database("rag_tutorial")

# 1. 先释放，清掉卡死状态
client.release_collection(collection_name="docs")
time.sleep(3)

# 2. 触发加载，但用短 timeout，不让它卡住（请求已发出，服务端会继续在后台加载）
try:
    client.load_collection(collection_name="docs", timeout=5)
except Exception as e:
    print(f"加载已在后台进行（这个提示可忽略）：{e}")

# 3. 轮询看进度，每 3 秒一次，最多等 ~4 分钟
for i in range(80):
    state = client.get_load_state(collection_name="docs")
    print(f"[{i*3:>3}s] {state}")
    if "Loaded" in str(state):      # 终于加载完
        print("✓ 加载完成！可以查了")
        break
    time.sleep(3)


2026-07-17 19:34:25,597 [WARNING][handler]: [wait_for_loading_collection] Retry timeout: 5s (decorators.py:327)
2026-07-17 19:34:25,601 [ERROR][_log_rpc_error]: RPC error: [wait_for_loading_collection], <MilvusException: (code=1, message=[wait_for_loading_collection] Retry timeout: 5s, message=wait for loading collection timeout, collection: docs)>, <elapsed:5121.5ms>
Traceback:
Traceback (most recent call last):
  File "e:\MiniConda3\envs\langchain1.2\Lib\site-packages\pymilvus\decorators.py", line 305, in handler
    return func(*args, **kwargs)
  File "e:\MiniConda3\envs\langchain1.2\Lib\site-packages\pymilvus\client\grpc_handler.py", line 1744, in wait_for_loading_collection
    raise MilvusException(
        message=f"wait for loading collection timeout, collection: {collection_name}"
    )
pymilvus.exceptions.MilvusException: <MilvusException: (code=1, message=wait for loading collection timeout, collection: docs)>

The above exception was the direct cause of the following except

加载已在后台进行（这个提示可忽略）：<MilvusException: (code=1, message=[load_collection] Retry timeout: 5s, message=[wait_for_loading_collection] Retry timeout: 5s, message=wait for loading collection timeout, collection: docs)>
[  0s] {'state': <LoadState: Loading>, 'progress': 50}
[  3s] {'state': <LoadState: Loading>, 'progress': 50}
[  6s] {'state': <LoadState: Loading>, 'progress': 50}
[  9s] {'state': <LoadState: Loading>, 'progress': 50}
[ 12s] {'state': <LoadState: Loading>, 'progress': 50}
[ 15s] {'state': <LoadState: Loading>, 'progress': 50}
[ 18s] {'state': <LoadState: Loading>, 'progress': 50}
[ 21s] {'state': <LoadState: Loading>, 'progress': 50}
[ 24s] {'state': <LoadState: Loading>, 'progress': 50}
[ 27s] {'state': <LoadState: Loading>, 'progress': 50}
[ 30s] {'state': <LoadState: Loading>, 'progress': 50}
[ 33s] {'state': <LoadState: Loading>, 'progress': 50}
[ 36s] {'state': <LoadState: Loading>, 'progress': 50}
[ 39s] {'state': <LoadState: Loading>, 'progress': 50}
[ 42s] {'state': <L

KeyboardInterrupt: 

In [8]:
import time
from pymilvus import MilvusClient
client = MilvusClient(uri="http://localhost:19530")
client.use_database("rag_tutorial")

client.load_collection(collection_name="docs", timeout=120)  # 直接等，给 2 分钟，别中断

state = client.get_load_state(collection_name="docs")
print("状态:", state)


2026-07-17 19:39:13,351 [WARNING][handler]: [wait_for_loading_collection] Retry timeout: 120s (decorators.py:327)
2026-07-17 19:39:13,355 [ERROR][_log_rpc_error]: RPC error: [wait_for_loading_collection], <MilvusException: (code=1, message=[wait_for_loading_collection] Retry timeout: 120s, message=wait for loading collection timeout, collection: docs)>, <elapsed:120177.3ms>
Traceback:
Traceback (most recent call last):
  File "e:\MiniConda3\envs\langchain1.2\Lib\site-packages\pymilvus\decorators.py", line 305, in handler
    return func(*args, **kwargs)
  File "e:\MiniConda3\envs\langchain1.2\Lib\site-packages\pymilvus\client\grpc_handler.py", line 1744, in wait_for_loading_collection
    raise MilvusException(
        message=f"wait for loading collection timeout, collection: {collection_name}"
    )
pymilvus.exceptions.MilvusException: <MilvusException: (code=1, message=wait for loading collection timeout, collection: docs)>

The above exception was the direct cause of the following 

MilvusException: <MilvusException: (code=1, message=[load_collection] Retry timeout: 120s, message=[wait_for_loading_collection] Retry timeout: 120s, message=wait for loading collection timeout, collection: docs)>

NameError: name 'COLLECTION_NAME' is not defined